# 🦾 RedRob — GRPO fine-tuning of Qwen3-0.6B

End-to-end notebook that takes the base **`Qwen/Qwen3-0.6B`** model, sets up the
RedRob candidate-ranking RL environment, GRPO-fine-tunes the policy against a
**rule-based reward model** (no LLM-as-a-judge), evaluates against a fixed
baseline rollout, saves all plots to `plots/`, and pushes the resulting
checkpoint to **`williyam/redrob-qwen-grpo`** on the Hugging Face Hub.

Hardware target: Apple M1 Pro 16 GB (MPS). Falls back to CUDA / CPU automatically.

---

**Repository layout this notebook assumes:**
```
redrob-reinforcement-learning/
├── configs/grpo_qwen3_0p6b.yaml
├── data/raw/sample_candidates.json
├── plots/
├── outputs/
└── src/redrob_rl/
```

Run every cell top to bottom. The training cell takes ~10 min on M1 Pro.

## 1. Imports and project paths

In [ ]:
import os, sys, json, time
from pathlib import Path

# Locate the project root (parent of `notebooks/`) and add `src/` to sys.path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC))

# Also expose the production Talentry-AI package (parent of redrob-reinforcement-learning)
TALENTRY_SRC = PROJECT_ROOT.parent / 'src'
if TALENTRY_SRC.exists():
    sys.path.insert(0, str(TALENTRY_SRC))

PROJECT_ROOT, SRC.exists(), TALENTRY_SRC.exists()

In [ ]:
import torch

def best_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

DEVICE = best_device()
print('Torch:', torch.__version__)
print('Device:', DEVICE)

## 2. Build the dataset of `(prompt, gold_decision, gold_score, context)` tuples

We use Talentry-AI's production ranker (BM25 + TF-IDF + behavioural signals
+ honeypot guard) to produce the gold labels. Candidates in the top-K are
labelled `"shortlist"`; everyone else is `"reject"`. Class balance is enforced.


In [ ]:
from redrob_rl import DatasetBuilder

builder = DatasetBuilder(
    candidates_path=PROJECT_ROOT / 'data' / 'raw' / 'sample_candidates.json',
    jd_path=PROJECT_ROOT / 'configs' / 'job_description.txt',
    top_k=15, max_samples=64, balance_classes=True, seed=7,
)
samples = builder.build()
print(f'built {len(samples)} samples '
      f'(positives: {sum(1 for s in samples if s.decision == "shortlist")})')

out_jsonl = builder.write_jsonl(samples, PROJECT_ROOT / 'data' / 'processed' / 'train.jsonl')
print('written:', out_jsonl)
samples[0].decision, samples[0].score, samples[0].candidate_id

## 3. Rule-based reward model and RL environment

Six interpretable components, each ∈ `[0, 1]`, weighted convex combination.

In [ ]:
from redrob_rl import RuleBasedRewardModel, CandidateRankEnv
from redrob_rl.env import rollout

reward_model = RuleBasedRewardModel()
env = CandidateRankEnv(samples, reward_model, sequential=True)

# Quick sanity check using a trivial policy
good = '{"decision":"shortlist","score":0.8,"reasons":["strong skill overlap","7 years YoE"]}'
bad = 'definitely no'
obs, info = env.reset(index=0)
print('gold:', info)
step = env.step(good)
print('reward (good completion):', round(step.reward, 3))
print('breakdown:', step.info['reward_breakdown'])


## 4. Load `Qwen/Qwen3-0.6B` and run the baseline rollout

Same env + same seed will be used after training for an apples-to-apples
comparison plot.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from redrob_rl.train import _make_policy, _strip_thinking, _format_chat

MODEL_NAME = 'Qwen/Qwen3-0.6B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float32, attn_implementation='eager'
).to(DEVICE)
print('loaded', MODEL_NAME, 'on', DEVICE)

In [ ]:
EVAL_N = 24
baseline_policy = _make_policy(base_model, tokenizer, DEVICE,
                               max_new_tokens=160, temperature=0.0)
base_eval = rollout(env, lambda p: _strip_thinking(baseline_policy(p)),
                    n_episodes=EVAL_N, seed=0)
print('baseline mean reward:', round(base_eval['mean_reward'], 4))
print('first decision (gold, completion[:120]):')
print(base_eval['decisions'][0])

## 5. Train with GRPO

We delegate to `redrob_rl.train.main(...)` so the notebook and the CLI run
identical code. The reward function uses our `RuleBasedRewardModel`.

In [ ]:
from redrob_rl import train as train_mod

CONFIG = str(PROJECT_ROOT / 'configs' / 'grpo_qwen3_0p6b.yaml')

# Free the base model we just loaded - train.py will reload it.
del base_model
import gc; gc.collect()
if DEVICE.type == 'mps':
    torch.mps.empty_cache()

metrics = train_mod.main(CONFIG, push_to_hub=False)
metrics

## 6. Inspect the saved plots

All four plots are written to `plots/` by `train_mod.main()`. We display
them inline here for convenience.

In [ ]:
from IPython.display import Image, display
for name in [
    'training_curves.png',
    'baseline_vs_trained.png',
    'reward_components.png',
    'reward_distribution.png',
]:
    p = PROJECT_ROOT / 'plots' / name
    print(p.name, p.exists())
    if p.exists():
        display(Image(filename=str(p)))

## 7. Push the fine-tuned checkpoint to the Hub

Requires the user to already be logged in via `huggingface_hub` (we use
the in-memory token, no `huggingface-cli` call needed).

In [ ]:
from huggingface_hub import whoami
whoami()['name']

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

OUT_DIR = PROJECT_ROOT / 'outputs' / 'redrob-qwen-grpo'
REPO_ID = 'williyam/redrob-qwen-grpo'

ft_tokenizer = AutoTokenizer.from_pretrained(str(OUT_DIR))
ft_model = AutoModelForCausalLM.from_pretrained(str(OUT_DIR), dtype=torch.float32)

ft_model.push_to_hub(REPO_ID, commit_message='GRPO fine-tune of Qwen3-0.6B with rule-based reward')
ft_tokenizer.push_to_hub(REPO_ID, commit_message='tokenizer')
print(f'pushed to https://huggingface.co/{REPO_ID}')

## 8. (Optional) reload from the Hub and sanity check

In [ ]:
tok_hub = AutoTokenizer.from_pretrained(REPO_ID)
mdl_hub = AutoModelForCausalLM.from_pretrained(REPO_ID, dtype=torch.float32).to(DEVICE)
policy_hub = _make_policy(mdl_hub, tok_hub, DEVICE, max_new_tokens=160, temperature=0.0)

env_check = CandidateRankEnv(samples, reward_model, sequential=True)
obs, info = env_check.reset(index=0)
completion = _strip_thinking(policy_hub(obs))
print('completion:', completion[:400])
step = env_check.step(completion)
print('reward:', round(step.reward, 3))
print('breakdown:', step.info['reward_breakdown'])